# Day 5: Counters, Shift Registers & Debouncing

## Pre-Class Videos (~45 minutes total)

| # | Segment | Duration | File |
|---|---------|----------|------|
| 1 | Counter Variations | ~10 min | `d05_s1_counter_variations.html` |
| 2 | Shift Registers | ~12 min | `d05_s2_shift_registers.html` |
| 3 | Metastability & Synchronizers | ~12 min | `d05_s3_metastability_synchronizers.html` |
| 4 | Button Debouncing | ~11 min | `d05_s4_button_debouncing.html` |

## Code Examples

| File | Description |
|------|-------------|
| `code/day05_ex01_counter_mod_n.v` | Parameterized modulo-N counter with `$clog2`, self-test |
| `code/day05_ex02_shift_reg_piso.v` | PISO shift register (UART TX building block), self-test |
| `code/day05_ex03_synchronizer.v` | 2-FF metastability synchronizer |
| `code/day05_ex04_debounce.v` | Counter-based debouncer with built-in sync, self-test |

## Diagrams

| File | Description |
|------|-------------|
| `diagrams/d05_synchronizer.svg` | 2-FF synchronizer schematic with metastability annotation |
| `diagrams/d05_bounce_waveform.svg` | Button bounce → debounced output waveform comparison |

## Key Concepts
- Counter variations: modulo-N, up/down, loadable
- Shift register types: SISO, SIPO, PISO, PIPO
- Metastability, setup/hold, and 2-FF synchronizers
- Counter-based debouncing, edge detection
- Input pipeline: sync → debounce → edge detect

## Directory Structure

```
day05_counters_shift_registers_debouncing/
├── d05_s1_counter_variations.html
├── d05_s2_shift_registers.html
├── d05_s3_metastability_synchronizers.html
├── d05_s4_button_debouncing.html
├── code/
│   ├── day05_ex01_counter_mod_n.v
│   ├── day05_ex02_shift_reg_piso.v
│   ├── day05_ex03_synchronizer.v
│   └── day05_ex04_debounce.v
├── diagrams/
│   ├── d05_synchronizer.svg
│   └── d05_bounce_waveform.svg
├── day05_quiz.md
└── day05_readme.md
```

---
## Code Examples

### `day05_ex01_counter_mod_n.v`

```verilog
// =============================================================================
// day05_ex01_counter_mod_n.v — Parameterized Modulo-N Counter
// Day 5: Counters, Shift Registers & Debouncing
// Accelerated HDL for Digital System Design · UCF ECE
// =============================================================================
// Build:  iverilog -o sim day05_ex01_counter_mod_n.v && vvp sim
// Synth:  yosys -p "read_verilog day05_ex01_counter_mod_n.v; synth_ice40 -top counter_mod_n"
// =============================================================================

module counter_mod_n #(
    parameter N = 10
)(
    input  wire                     i_clk,
    input  wire                     i_reset,
    input  wire                     i_enable,
    output reg  [$clog2(N)-1:0]     o_count,
    output wire                     o_wrap
);

    always @(posedge i_clk) begin
        if (i_reset)
            o_count <= 0;
        else if (i_enable) begin
            if (o_count == N - 1)
                o_count <= 0;
            else
                o_count <= o_count + 1;
        end
    end

    assign o_wrap = i_enable && (o_count == N - 1);

endmodule

// =============================================================================
// Quick self-test (simulation only)
// =============================================================================
`ifdef SIMULATION
module tb_counter_mod_n;
    reg clk = 0, reset = 1, enable = 0;
    wire [3:0] count;
    wire       wrap;

    counter_mod_n #(.N(10)) uut (
        .i_clk(clk), .i_reset(reset), .i_enable(enable),
        .o_count(count), .o_wrap(wrap)
    );

    always #20 clk = ~clk;

    integer fail_count = 0;

    initial begin
        $dumpfile("tb_counter_mod_n.vcd");
        $dumpvars(0, tb_counter_mod_n);

        #100; reset = 0; enable = 1;

        // Let it count through two full cycles (20 clocks for mod-10)
        repeat (20) @(posedge clk);
        #1;
        if (count !== 0) begin
            $display("FAIL: Expected 0 after 20 clocks, got %0d", count);
            fail_count = fail_count + 1;
        end else
            $display("PASS: Counter wrapped correctly after 20 clocks");

        // Check wrap pulse
        repeat (8) @(posedge clk);  // count = 8
        @(posedge clk); #1;        // count = 9
        if (wrap !== 1) begin
            $display("FAIL: wrap should be 1 at count 9");
            fail_count = fail_count + 1;
        end else
            $display("PASS: wrap asserted at count 9");

        if (fail_count == 0) $display("\n*** ALL TESTS PASSED ***");
        else $display("\n*** %0d FAILURES ***", fail_count);
        $finish;
    end
endmodule
`endif
```

### `day05_ex02_shift_reg_piso.v`

```verilog
// =============================================================================
// day05_ex02_shift_reg_piso.v — Parallel-In Serial-Out Shift Register
// Day 5: Counters, Shift Registers & Debouncing
// Accelerated HDL for Digital System Design · UCF ECE
// =============================================================================
// This is the core building block for UART TX (Week 3).
// Load a byte, then shift it out LSB-first one bit per clock.
// =============================================================================
// Build:  iverilog -o sim day05_ex02_shift_reg_piso.v && vvp sim
// Synth:  yosys -p "read_verilog day05_ex02_shift_reg_piso.v; synth_ice40 -top shift_reg_piso"
// =============================================================================

module shift_reg_piso #(
    parameter WIDTH = 8
)(
    input  wire              i_clk,
    input  wire              i_reset,
    input  wire              i_load,
    input  wire              i_shift_en,
    input  wire [WIDTH-1:0]  i_parallel_in,
    output wire              o_serial_out
);

    reg [WIDTH-1:0] r_shift;

    always @(posedge i_clk) begin
        if (i_reset)
            r_shift <= {WIDTH{1'b0}};
        else if (i_load)
            r_shift <= i_parallel_in;       // load wins over shift
        else if (i_shift_en)
            r_shift <= {1'b0, r_shift[WIDTH-1:1]};  // shift right, fill MSB with 0
    end

    assign o_serial_out = r_shift[0];       // LSB first (UART convention)

endmodule

// =============================================================================
// Quick self-test
// =============================================================================
`ifdef SIMULATION
module tb_shift_reg_piso;
    reg        clk = 0, reset = 1, load = 0, shift_en = 0;
    reg  [7:0] parallel_in;
    wire       serial_out;

    shift_reg_piso #(.WIDTH(8)) uut (
        .i_clk(clk), .i_reset(reset),
        .i_load(load), .i_shift_en(shift_en),
        .i_parallel_in(parallel_in),
        .o_serial_out(serial_out)
    );

    always #20 clk = ~clk;

    integer i, fail_count = 0;
    reg [7:0] test_byte;
    reg expected_bit;

    initial begin
        $dumpfile("tb_shift_reg_piso.vcd");
        $dumpvars(0, tb_shift_reg_piso);

        #100; reset = 0;
        test_byte = 8'hA5;  // 10100101 — good mix of 0s and 1s

        // Load the byte
        @(posedge clk);
        parallel_in = test_byte;
        load = 1;
        @(posedge clk);
        load = 0;
        shift_en = 1;

        // Shift out 8 bits and verify LSB-first
        for (i = 0; i < 8; i = i + 1) begin
            expected_bit = test_byte[i];
            #1;
            if (serial_out !== expected_bit) begin
                $display("FAIL: Bit %0d — expected %b, got %b", i, expected_bit, serial_out);
                fail_count = fail_count + 1;
            end else
                $display("PASS: Bit %0d = %b", i, serial_out);
            @(posedge clk);
        end

        if (fail_count == 0) $display("\n*** ALL TESTS PASSED ***");
        else $display("\n*** %0d FAILURES ***", fail_count);
        $finish;
    end
endmodule
`endif
```

### `day05_ex03_synchronizer.v`

```verilog
// =============================================================================
// day05_ex03_synchronizer.v — 2-FF Metastability Synchronizer
// Day 5: Counters, Shift Registers & Debouncing
// Accelerated HDL for Digital System Design · UCF ECE
// =============================================================================
// Use this for EVERY signal entering your clock domain from outside:
//   - Button/switch inputs
//   - External serial data lines (before any processing)
//   - Signals from other clock domains
// =============================================================================
// Synth:  yosys -p "read_verilog day05_ex03_synchronizer.v; synth_ice40 -top synchronizer"
// =============================================================================

module synchronizer (
    input  wire i_clk,
    input  wire i_async_in,
    output wire o_sync_out
);

    reg r_meta;     // First FF — may go metastable
    reg r_sync;     // Second FF — extremely unlikely to be metastable

    always @(posedge i_clk) begin
        r_meta <= i_async_in;    // Stage 1: might go metastable
        r_sync <= r_meta;        // Stage 2: gives stage 1 a full period to resolve
    end

    assign o_sync_out = r_sync;

endmodule
```

### `day05_ex04_debounce.v`

```verilog
// =============================================================================
// day05_ex04_debounce.v — Counter-Based Button Debouncer
// Day 5: Counters, Shift Registers & Debouncing
// Accelerated HDL for Digital System Design · UCF ECE
// =============================================================================
// Includes built-in 2-FF synchronizer. Connect directly to raw button input.
// Pipeline: async_in → [2-FF sync] → [debounce counter] → clean output
// =============================================================================
// Build:  iverilog -DSIMULATION -o sim day05_ex04_debounce.v && vvp sim
// Synth:  yosys -p "read_verilog day05_ex04_debounce.v; synth_ice40 -top debounce"
// =============================================================================

module debounce #(
    parameter CLKS_TO_STABLE = 250_000  // 10 ms at 25 MHz (override for sim)
)(
    input  wire i_clk,
    input  wire i_bouncy,
    output reg  o_clean
);

    // ---- 2-FF Synchronizer (built-in) ----
    reg r_sync_0, r_sync_1;
    always @(posedge i_clk) begin
        r_sync_0 <= i_bouncy;
        r_sync_1 <= r_sync_0;
    end

    // ---- Debounce Counter ----
    reg [$clog2(CLKS_TO_STABLE)-1:0] r_count;

    always @(posedge i_clk) begin
        if (r_sync_1 != o_clean) begin
            // Input differs from output — count how long
            if (r_count == CLKS_TO_STABLE - 1) begin
                o_clean <= r_sync_1;    // accept new value
                r_count <= 0;
            end else
                r_count <= r_count + 1;
        end else begin
            // Input matches output — reset counter
            r_count <= 0;
        end
    end

endmodule

// =============================================================================
// Self-test: simulate noisy button input
// =============================================================================
`ifdef SIMULATION
module tb_debounce;
    reg clk = 0, bouncy = 1;
    wire clean;

    // Short threshold for simulation (10 cycles instead of 250K)
    debounce #(.CLKS_TO_STABLE(10)) uut (
        .i_clk(clk), .i_bouncy(bouncy), .o_clean(clean)
    );

    always #20 clk = ~clk;  // 25 MHz

    integer fail_count = 0;

    task bounce_press;
        // Simulate 5 bounces then stable low
        integer i;
        begin
            for (i = 0; i < 5; i = i + 1) begin
                bouncy = 0; repeat (2) @(posedge clk);
                bouncy = 1; repeat (2) @(posedge clk);
            end
            bouncy = 0;  // stable press
        end
    endtask

    initial begin
        $dumpfile("tb_debounce.vcd");
        $dumpvars(0, tb_debounce);

        // Wait for initial state to settle
        repeat (5) @(posedge clk);

        // Should start high (idle)
        #1;
        if (clean !== 1'bx && clean !== 1'b1) begin
            // Accept x or 1 at startup
        end

        // Press with bounces
        bounce_press;
        // Wait for debounce threshold + sync latency
        repeat (20) @(posedge clk);
        #1;
        if (clean !== 1'b0) begin
            $display("FAIL: clean should be 0 after stable press");
            fail_count = fail_count + 1;
        end else
            $display("PASS: Debounced press detected");

        // Hold stable
        repeat (20) @(posedge clk);

        // Release (clean, no bounce for simplicity)
        bouncy = 1;
        repeat (20) @(posedge clk);
        #1;
        if (clean !== 1'b1) begin
            $display("FAIL: clean should be 1 after release");
            fail_count = fail_count + 1;
        end else
            $display("PASS: Debounced release detected");

        if (fail_count == 0) $display("\n*** ALL TESTS PASSED ***");
        else $display("\n*** %0d FAILURES ***", fail_count);
        $finish;
    end
endmodule
`endif
```

---
## Pre-Class Self-Check Quiz

**Q1:** What does `$clog2(N)` return and why is it useful for counter design?

<details><summary>Answer</summary>
`$clog2(N)` returns ceil(log₂(N)) — the number of bits needed to represent values 0 through N−1. It auto-sizes counter widths so that when you change the parameter N, the bit width adjusts automatically. No manual calculation needed.
</details>

**Q2:** What type of shift register is the core of a UART transmitter? A UART receiver?

<details><summary>Answer</summary>
UART TX: **PISO** (Parallel In, Serial Out) — load a byte, shift it out one bit at a time.
UART RX: **SIPO** (Serial In, Parallel Out) — shift bits in one at a time, present the complete byte.
</details>

**Q3:** What causes metastability and what is the standard mitigation?

<details><summary>Answer</summary>
Metastability occurs when an asynchronous signal (not synchronized to the clock) violates setup/hold timing of a flip-flop. The standard mitigation is a **2-FF synchronizer**: two flip-flops in series. The first may go metastable but has a full clock period to resolve before the second samples it.
</details>

**Q4:** Why must you synchronize *before* debouncing, not the other way around?

<details><summary>Answer</summary>
The debounce counter uses flip-flops clocked by your system clock. If the input is asynchronous (not synchronized), those flip-flops can go metastable. Synchronize first to make the signal safe for all downstream clocked logic, then debounce the clean synchronized signal.
</details>